# Week 10, Day 6: Forge: Week 10 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1.", 2 ** 3, "(an antiderivative is t^3)")
print("2.", np.convolve([1, 1], [1, 2, 3]))
print("3.", round(20 * (1 - np.exp(-1)), 2), "mV")

In [ ]:
rng = np.random.default_rng(2001)
P, L, T = 16, 15, 30000                       # pixels, lags (150 ms), frames
x = np.arange(P) - 7.5
k = np.arange(L)
_sp = np.exp(-x ** 2 / (2 * 1.5 ** 2)) - 0.5 * np.exp(-x ** 2 / (2 * 4 ** 2)); _sp /= np.linalg.norm(_sp)
_tm = np.exp(-(k - 3) ** 2 / 3) - 0.6 * np.exp(-(k - 7) ** 2 / 6); _tm /= np.linalg.norm(_tm)
_RF = np.outer(_sp, _tm)                      # hidden: pixels by lags

def generator(S, F):
    """Generator signal of a space-time filter F (pixels by lags) applied to stimulus S (frames by pixels)."""
    g = np.zeros(len(S))
    for lag in range(F.shape[1]):
        g[lag:] += S[:len(S) - lag] @ F[:, lag]
    return g

S = rng.normal(size=(T, P))
spikes = rng.poisson(0.05 * np.exp(1.5 * generator(S, _RF)))
S_rep = rng.normal(size=(500, P))
_rate_rep = 0.05 * np.exp(1.5 * generator(S_rep, _RF))
repeats = rng.poisson(_rate_rep, size=(40, 500))   # 40 repeats of the test stimulus
print("training spikes:", spikes.sum(), "  test spikes per repeat:", repeats.sum(axis=1).mean())

### Exercise: Part A: the spatiotemporal receptive field

Compute the spike-triggered average for every pixel and lag (a 16 by 15 array; one matrix product per lag). Take its SVD: store the separability index $\sigma_1^2/\sum\sigma_i^2$ (Week 7) in `sep_index`, and the top left and right singular vectors, the spatial and temporal profiles, in `spatial_hat` and `temporal_hat`. Plot the STA and both profiles.

In [ ]:
sta = np.zeros((P, L))
for lag in range(L):
    sta[:, lag] = ...

U, s, Vt = np.linalg.svd(sta)
sep_index = ...
spatial_hat, temporal_hat = ..., ...

fig, ax = plt.subplots(1, 3, figsize=(10, 2.8))
ax[0].imshow(sta, cmap="RdBu_r", aspect="auto"); ax[0].set_xlabel("lag (10 ms)"); ax[0].set_ylabel("pixel"); ax[0].set_title("STA")
ax[1].plot(x, spatial_hat * np.sign(spatial_hat[8])); ax[1].set_title("spatial profile"); ax[1].set_xlabel("pixel")
ax[2].plot(k * 10, temporal_hat * np.sign(temporal_hat[3])); ax[2].set_title("temporal profile"); ax[2].set_xlabel("ms before spike")
plt.tight_layout(); plt.show()
print("separability", round(sep_index, 3))

In [ ]:
import numpy as np
def test_sta():
    "sta is pixels by lags and matches the definition at one entry"
    assert sta.shape == (P, L)
    assert np.isclose(sta[5, 3], np.sum(spikes[L:] * S[L - 3:T - 3, 5]) / spikes[L:].sum())

def test_separable():
    "the receptive field is close to separable: sep_index above 0.9"
    assert sep_index > 0.9

def test_profiles():
    "the SVD's top spatial and temporal vectors match the hidden profiles"
    assert abs(spatial_hat @ _sp) > 0.97 and abs(temporal_hat @ _tm) > 0.97

test_sta()
test_separable()
test_profiles()
print("All checks passed")

### Exercise: Part B: measure the nonlinearity and predict the repeats

Compute the training generator signal with the unit-length STA as the filter, split it into 20 quantile bins, and store the mean spike count (`nl`) and mean generator value (`centres`) of each bin. Then predict the test responses: compute the generator for `S_rep`, interpolate the nonlinearity with `np.interp(g, centres, nl)` to get `pred`, and store its correlation with the repeat-averaged PSTH in `r_model`.

In [ ]:
F_hat = sta / np.linalg.norm(sta)
g_train = generator(S, F_hat)
edges = np.quantile(g_train, np.linspace(0, 1, 21))
which = np.clip(np.digitize(g_train, edges) - 1, 0, 19)
nl = ...
centres = ...

pred = ...
psth = repeats.mean(axis=0)
r_model = ...

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(centres, nl, "o-"); ax[0].set_xlabel("generator signal"); ax[0].set_ylabel("spikes per frame"); ax[0].set_title("measured nonlinearity")
ax[1].plot(np.arange(200) * 10, psth[:200], label="PSTH (40 repeats)"); ax[1].plot(np.arange(200) * 10, pred[:200], label="model")
ax[1].set_xlabel("time (ms)"); ax[1].legend(frameon=False); plt.tight_layout(); plt.show()
print("correlation of model with PSTH:", round(r_model, 3))

In [ ]:
import numpy as np
def test_nl():
    "nl and centres describe the nonlinearity in 20 bins, rising with the generator"
    assert nl.shape == (20,) and centres.shape == (20,)
    assert nl[-1] > 5 * nl[0] and np.all(np.diff(centres) > 0)

def test_pred():
    "pred has one predicted mean count per test frame and correlates with the PSTH above 0.7"
    assert pred.shape == (500,)
    assert r_model > 0.7

test_nl()
test_pred()
print("All checks passed")

### Exercise: Part C: the noise ceiling

Compute `r_half` from even and odd repeats, the corrected reliability `r_full`, and the model's normalised correlation, `normalised` = `r_model` divided by the ceiling $\sqrt{r_{\text{full}}}$. How much of the explainable response does the LN model capture?

In [ ]:
r_half = ...
r_full = ...
normalised = ...
print(f"split-half {r_half:.3f}, reliability {r_full:.3f}, ceiling {np.sqrt(r_full):.3f};  model {r_model:.3f} = {100 * normalised:.0f}% of the ceiling")

In [ ]:
import numpy as np
def test_reliability():
    "r_half and r_full follow the definitions"
    A, B = repeats[::2].mean(axis=0), repeats[1::2].mean(axis=0)
    assert np.isclose(r_half, np.corrcoef(A, B)[0, 1])
    assert np.isclose(r_full, 2 * r_half / (1 + r_half))

def test_normalised():
    "the model reaches more than 70% of the ceiling"
    assert np.isclose(normalised, r_model / np.sqrt(r_full))
    assert normalised > 0.7

test_reliability()
test_normalised()
print("All checks passed")